In [3]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor

# 학습 모델 저장을 위한 라이브러리
import pickle

In [4]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = 'model/best_model_finalproject_stratifiedKfold.dat'

# 교차검증 횟수
cv_count = 10

# 교차 검증
#kfold = KFold(n_splits=cv_count, shuffle=True, random_state=1)

# Stratified K-Fold 설정 (클래스 비율 유지)
kfold = StratifiedKFold(n_splits=cv_count, shuffle=True, random_state=1)


# 평가 결과를 담을 리스트
f1_score_list = []

# 학습 모델 이름
model_name_list = []

In [5]:
# 데이터 준비
df = pd.read_csv("C:/Users/HR/Desktop/workspace/파이널프로젝트/train_wide.csv") # 파일 Path 입력

# 입력과 결과로 나눈다.
#X = df.drop(columns=['ID', 'Segment', '기준년월'])
X = df.drop(columns=['ID', 'Segment']) #wide 데이터에는 기준년월 컬럼이 없음.
y = df['Segment']

In [6]:
# 문자열 -> 숫자
encoder1 = LabelEncoder()
encoder1.fit(y)
y2 = encoder1.transform(y)

obj_cols = X.select_dtypes(include='object').columns

# 각 컬럼에 대해 Label Encoding 수행
for col in obj_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

In [7]:
# 입력 데이터 표준화
scaler1 = StandardScaler()
scaler1.fit(X)
X2 = scaler1.transform(X)

In [8]:
# 학습할 데이터를 변수에 담아준다.
train_X = X2
train_y = y2

acc_score_list = []

In [9]:
# XGBoost 하이퍼파라미터 튜닝

params = {
    'booster' : ['gblinear'],
    'n_estimators' : [50, 100, 150, 200, 250, 300],
    'learning_rate': [0.01]
}

temp_model = XGBClassifier(verbosity=0, use_label_encoder=False, tree_method='gpu_hist', gpu_id=0)
xgboost_grid_clf = GridSearchCV(temp_model, param_grid=params, scoring='f1', cv=kfold)
xgboost_grid_clf.fit(train_X, train_y)

# 평가 결과를 담아준다.
f1_score_list.append(xgboost_grid_clf.best_score_)
# 학습 모델 이름을 담아준다.
model_name_list.append("XGBoost Tuning 0.01 gblinear")
print(xgboost_grid_clf.best_params_) # booster랑 n_estimators 조합
print(xgboost_grid_clf.best_score_) # 스코어

KeyboardInterrupt: 